# Data modeling 3: slowly changing dimensions

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-03-slowly-changing-dimensions.ipynb)

A customer moves from SMB to enterprise. Should last month's sale appear under the old segment or the current one? Write down which question the report answers before deciding whether to overwrite or create a new version.

Dimension attributes change: customers move, change segment, get a new account manager. A **slowly changing
dimension** (SCD) is a rule for what the dimension does when that happens. Overwrite the old value and every
past fact is reported with today's attribute. Keep history and every fact can be reported with the value that
was true when it happened.

This lesson builds the shop's customer dimension with full history from the CRM's audit log, joins facts to
the version that was current at the time, loads the next day's changes incrementally, and tests the result.

**Prerequisites:** [Data modeling 2: star schema design](data-modeling-02-star-schema-design.ipynb), and window functions ([SQL 8](../sql/sql-08-window-functions.ipynb)).

*Last reviewed: November 2026, against Apache Spark 4.0 and Delta Lake 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session with Delta Lake. The second loads `src_customers` (the CRM's current view of each customer), `src_customer_changes` (the CRM's audit log of segment and city changes), `src_orders` (one row per order, with fulfilment timestamps) and `src_order_lines` (one row per product on an order) as Delta tables: extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import os
import re
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["TZ"] = "UTC"  # timestamps collected into Python stay in UTC, like the session below
time.tzset()

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.sources.default", "delta")  # `create table` makes Delta tables
    .config("spark.sql.warehouse.dir", tempfile.mkdtemp(prefix="modeling-"))  # where the tables are stored
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("FATAL")  # quiet Spark's own logging; errors still raise in Python


def run(script):
    """Run several SQL statements that each end with a semicolon: spark.sql() takes one statement at a time."""
    for statement in re.split(r";[ \t]*(?:--[^\n]*)?(?:\n|$)", script):
        if statement.strip():
            spark.sql(statement)


print("Spark", spark.version)

In [ ]:
run("""
create table src_customers (
    customer_id int, name string, email string, city string, segment string, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_customer_changes (customer_id int, changed_at timestamp, city string, segment string);
insert into src_customer_changes values  -- the CRM's audit log: one row each time a customer's details changed
    (101, '2025-11-02 00:00', 'Leeds', 'smb'),
    (102, '2025-11-20 00:00', 'York',  'smb'),
    (102, '2026-02-01 00:00', 'York',  'enterprise'),
    (103, '2025-12-05 00:00', 'Bath',  'enterprise'),
    (104, '2026-01-08 00:00', 'Hull',  'smb'),
    (104, '2026-02-15 00:00', 'Leeds', 'smb'),
    (105, '2026-01-19 00:00', 'Derby', 'smb'),
    (106, '2026-02-10 00:00', 'York',  'mid-market');
create table src_orders (
    order_id int, customer_id int, order_ts timestamp, channel string, status string,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku string, qty int, unit_price decimal(8, 2), discount decimal(8, 2)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
""")
spark.sql("show tables").show()

The cell below builds `fact_sales` at order-line grain keyed by the source `customer_id`, as in lesson 1, so the
examples can focus on the customer dimension.

In [ ]:
spark.sql("""
create table fact_sales as
select l.order_id, l.line_no, o.customer_id, o.order_ts, l.qty * l.unit_price - l.discount as net_amount
from src_order_lines l join src_orders o using (order_id)
where o.status <> 'cancelled'
""")
spark.sql("select customer_id, count(*) as lines, sum(net_amount) as net_sales from fact_sales group by all order by all").show(truncate=False)

## Why history matters

Customer 102 was an `smb` customer until February 1, when it became `enterprise`. It placed an order in January
and another in February. Finance asks: how much did each segment buy in January?

If the dimension simply holds the CRM's current values (an overwrite, **Type 1**), customer 102 is
`enterprise` everywhere, and its January order moves from `smb` to `enterprise` after the fact. Reports for
January change every time a customer changes segment, even though January is long over. Sometimes that is what
you want (a corrected typo should be corrected everywhere). For segment, region, price tier or sales owner it
usually is not.

**Where it matters**
- Any attribute used to attribute revenue or performance: segment, territory, account owner, price list, risk tier.
- Audit and regulatory reporting, where past reports must be reproducible.

**Watch out for**
- Discovering the need for history after a year of overwrites: the old values are gone unless a source kept them.

### See what a current-state join changes

Filter the orders to January, then join to today’s customer segment. A customer who upgraded in February appears under the new segment even for a January order.

In [ ]:
spark.sql("""
select c.segment, sum(f.net_amount) as january_sales
from fact_sales f join src_customers c using (customer_id)
where f.order_ts < '2026-02-01'
group by all order by all
""").show(truncate=False)

**What to notice:** January's 292.50 of enterprise sales include customer 102's 60.00 order, placed while 102 was still `smb`.
With overwritten attributes there is no way to recover the split that finance reported at the end of January.

## The SCD types

| Type | On a change | History | Typical attributes |
|---|---|---|---|
| **0** | keep the original value | none, frozen | sign-up date, original channel |
| **1** | overwrite in place | none | corrections, email, phone |
| **2** | close the current row, add a new row with a new surrogate key | full | segment, territory, account owner |
| **3** | move the old value to a `previous_` column | one prior value | a planned reorganization |
| **4** | keep current values in the dimension, history in a separate table | full, kept apart | fast-changing attributes |
| **6** | Type 2 rows plus a Type 1 `current_` column on every row | full, plus today's value | when reports need both views |

Choose the type **per attribute**, not per table. A customer dimension is typically Type 2 on segment and
city and Type 1 on email: fixing an email address should not create a new version. A Type 2 row carries
`valid_from`, `valid_to` and `is_current`. The range is **half-open**: a version is valid from `valid_from`
inclusive up to `valid_to` exclusive, and the current version's `valid_to` is a far-future date such as
`9999-12-31`, not null, so range conditions need no special case.

**Where it matters**
- Designing every dimension: list its attributes and give each a type before writing the load.

**Watch out for**
- Making every attribute Type 2: a frequently corrected column creates a new version per correction and the dimension balloons.
- A null `valid_to` for current rows: every range condition then needs `or valid_to is null`, and someone forgets it.

## Building a Type 2 dimension from a change log

When the source keeps an audit log (or you have kept daily snapshots), history can be built in one query. Each
change row starts a version; the version ends where the next change for the same customer begins.
`lead(changed_at)` over the customer's changes gives that end. The last version has no next change, so its end
is `9999-12-31`, and it is the current row.

**Where it matters**
- Initial loads and rebuilds from CDC logs, audit tables or a history of snapshots.

**Watch out for**
- Several changes at the same timestamp for one key: deduplicate first, or two versions have zero length.
- Rows where nothing tracked changed (only an untracked column did): drop them, or you get versions identical to their predecessor.

### Turn customer changes into date ranges

Within each customer, use the next change timestamp as the current version’s end. Give each version a warehouse key. Keep segment and city from the historical row and name and email from today’s CRM record.

In [ ]:
spark.sql("""
create table dim_customer as
select row_number() over (order by h.customer_id, h.changed_at)                         as customer_sk,
       h.customer_id,
       c.name                                                                         as customer_name,  -- Type 1
       c.email,                                                                                          -- Type 1
       h.city,                                                                                           -- Type 2
       h.segment,                                                                                        -- Type 2
       h.changed_at                                                                   as valid_from,
       coalesce(lead(h.changed_at) over w, timestamp '9999-12-31')                    as valid_to,
       lead(h.changed_at) over w is null                                              as is_current
from src_customer_changes h
join src_customers c using (customer_id)
window w as (partition by h.customer_id order by h.changed_at)
""")
spark.sql("""
select customer_sk, customer_id, city, segment, valid_from, valid_to, is_current from dim_customer order by customer_sk
""").show(truncate=False)

**What to notice:** Customers 102 and 104 have two rows each; everyone else has one. Customer 102's `smb` version ends at exactly
the instant its `enterprise` version begins (2026-02-01 00:00), and only the newer row has `is_current = true`.
Each version has its own surrogate key, so a fact can point at "102 as smb" (key 2) or "102 as enterprise" (key 3).

## Point-in-time joins

 To report with the value that was true at the time, join each fact to the version whose range contains the
 fact's timestamp:

 ```sql
 on d.customer_id = f.customer_id
and f.order_ts >= d.valid_from
and f.order_ts <  d.valid_to
 ```

 The half-open range matters. With `between valid_from and valid_to`, a fact at exactly the boundary instant
 matches both the old and the new version, and the join duplicates it.

 You can do this lookup **when the fact is loaded**, storing the version's surrogate key on the fact (the
 classic Kimball approach: reports then join on one integer key), or **at query time** from the natural key and
 timestamp (simpler loads, more expensive queries, and corrections to history apply immediately). Most
 warehouses resolve at load time for the main facts.

**Where it matters**
- Revenue by the segment, region or owner at the time of sale; reproducing last quarter's report today.

**Watch out for**
- `between` on validity ranges: boundary instants double count.
- Joining on the natural key alone to a Type 2 dimension: every fact matches every version of its customer.

**Quick check.** If the new version begins on February 1, a sale exactly at midnight belongs to that new version. Write the predicate as `valid_from <= sale_time AND sale_time < valid_to`.

### Resolve the customer as of the order

Join on customer ID and require the order timestamp to fall in the version’s validity range. Compare with the ID-only join, which can match several versions and inflate sales.

In [ ]:
spark.sql("""
select d.segment, sum(f.net_amount) as january_sales
from fact_sales f
join dim_customer d
  on d.customer_id = f.customer_id and f.order_ts >= d.valid_from and f.order_ts < d.valid_to
where f.order_ts < '2026-02-01'
group by all order by all
""").show(truncate=False)

In [ ]:
spark.sql("""
select 'point-in-time join' as join_used, count(*) as fact_rows, sum(f.net_amount) as net_sales
from fact_sales f join dim_customer d
  on d.customer_id = f.customer_id and f.order_ts >= d.valid_from and f.order_ts < d.valid_to
union all
select 'natural key only', count(*), sum(f.net_amount)
from fact_sales f join dim_customer d on d.customer_id = f.customer_id
""").show(truncate=False)

**What to notice:** Now customer 102's January order counts as `smb`: enterprise January sales drop to 232.50 and smb rises
from 102.00 to 162.00. The January total is the same 394.50; only the attribution moved. The second table shows why the date condition is not optional: joining on
`customer_id` alone matches each fact of customers 102 and 104 to both of their versions, turning 14 fact rows
into 20 and sales of 1,017.50 into 1,613.50.

### Assign the boundary to one version

If one version ends at midnight and the next starts at midnight, inclusive BETWEEN can match both. Use an inclusive start and exclusive end so the new version wins.

In [ ]:
spark.sql("""
with boundary_order as (
    select * from values (9001, 102, timestamp '2026-02-01 00:00', 50.00) as t(order_id, customer_id, order_ts, net_amount)
)
select 'between (inclusive end)' as range_rule, count(*) as matched_versions, collect_list(d.segment) as segments
from boundary_order f join dim_customer d
  on d.customer_id = f.customer_id and f.order_ts between d.valid_from and d.valid_to
union all
select 'half-open [from, to)', count(*), collect_list(d.segment)
from boundary_order f join dim_customer d
  on d.customer_id = f.customer_id and f.order_ts >= d.valid_from and f.order_ts < d.valid_to
""").show(truncate=False)

**What to notice:** With `between`, the order at 2026-02-01 00:00 matches the `smb` row (whose `valid_to` is that instant) *and* the
`enterprise` row: one order, two rows, double the sales. The half-open rule gives exactly one version,
`enterprise`, because the change was effective from that instant.

## Loading changes incrementally

After the initial build, each load compares a new snapshot of the source with the current dimension rows:

1. **Stage** the snapshot, one row per natural key.
2. **Detect changes** in the Type 2 attributes with a null-safe comparison: `is distinct from` treats
   `null → 'Leeds'` as a change, where `<>` returns null and misses it.
3. **Close** each changed current row: set `valid_to` to the load time and `is_current` to false.
4. **Insert** a new current version for each changed key and each new key.
5. **Overwrite** Type 1 attributes on all versions of the key.

Steps 3 and 4 must commit together, or readers can see a customer with no current row or two. Delta Lake has no
multi-statement transactions, but every single statement is atomic, so on Delta (and in most warehouses) the
close and the insert are written as **one `merge`**. The load must also be **idempotent**: running it twice
with the same snapshot must change nothing the second time, because retries happen.

The `merge` needs one trick. A `merge` updates *or* inserts for each source row, but a changed customer needs
both: close the old version and insert a new one. So the source lists each changed customer twice: once
with its key, which matches and closes the current row, and once with a null key, which never matches and is
therefore inserted as the new version.

**Where it matters**
- Daily or hourly dimension loads from snapshots or CDC.

**Watch out for**
- Using the load time as `valid_from` when the source tells you when the change happened: prefer the source's effective time.
- Comparing with `<>`: changes to or from null are missed.

### Apply three kinds of customer change

Customer 101 changes segment, so close the old version and create a new one. Customer 105 changes email, so update the Type 1 attribute. Customer 107 is new, so insert its first version.

In [ ]:
run("""
create table stg_customers as select * from src_customers;
update stg_customers set segment = 'mid-market' where customer_id = 101;
update stg_customers set email = 'sales@echo.example' where customer_id = 105;
insert into stg_customers values (107, 'Gamma Goods', 'gg@gamma.example', 'Bath', 'smb', date '2026-02-28');
""")

SCD2_SOURCE = """
create or replace table scd2_source as                 -- what the merge will do, one row per action
with staged as (
    select s.*, c.customer_sk is null as is_new,
           c.segment is distinct from s.segment or c.city is distinct from s.city as is_changed
    from stg_customers s left join dim_customer c on c.customer_id = s.customer_id and c.is_current
),
new_versions as (                                      -- rows to insert get the next surrogate keys
    select *, (select max(customer_sk) from dim_customer) + row_number() over (order by customer_id) as new_sk
    from staged where is_new or is_changed
)
select customer_id as merge_key, customer_id, name, email, city, segment, cast(null as bigint) as new_sk
from staged where is_changed and not is_new            -- matches the current row and closes it
union all
select null, customer_id, name, email, city, segment, new_sk
from new_versions                                      -- a null key never matches: inserted as the new version
"""

SCD2_MERGE = """
merge into dim_customer d
using scd2_source s
on d.customer_id = s.merge_key and d.is_current
when matched then
    update set valid_to = timestamp '2026-03-01', is_current = false
when not matched then
    insert (customer_sk, customer_id, customer_name, email, city, segment, valid_from, valid_to, is_current)
    values (s.new_sk, s.customer_id, s.name, s.email, s.city, s.segment,
            timestamp '2026-03-01', timestamp '9999-12-31', true)
"""

TYPE1_MERGE = """
merge into dim_customer d
using stg_customers s
on d.customer_id = s.customer_id
when matched and (d.customer_name is distinct from s.name or d.email is distinct from s.email) then
    update set customer_name = s.name, email = s.email          -- every version, current or not
"""

spark.sql(SCD2_SOURCE)
spark.table("scd2_source").orderBy("customer_id", "merge_key").show(truncate=False)
spark.sql(SCD2_MERGE)
spark.sql(TYPE1_MERGE)
spark.sql("""
select customer_sk, customer_id, email, city, segment, valid_from, valid_to, is_current
from dim_customer where customer_id in (101, 105, 107) order by customer_sk
""").show(truncate=False)

In [ ]:
for statement in (SCD2_SOURCE, SCD2_MERGE, TYPE1_MERGE):  # the same snapshot again, as a retry would do
    spark.sql(statement)
(
    spark.sql("describe history dim_customer")
    .selectExpr("version", "operation",
                "operationMetrics.numTargetRowsUpdated as rows_updated",
                "operationMetrics.numTargetRowsInserted as rows_inserted")
    .orderBy("version")
    .show(truncate=False)
)

**What to notice:** The merge source lists customer 101 twice: with `merge_key` 101, which closes its current row, and with a null
key and new key 9, which inserts the new version. Customer 107 appears once, with a null key: it is only
inserted. Customer 101's `smb` row was closed at 2026-03-01 and a `mid-market` row (key 9) opened at the same instant.
Customer 105 still has one row: the new email was overwritten in place because email is Type 1. Customer 107
arrived as a new current row (key 10). Delta's table history records every write: the Type 2 `merge` (version 1)
updated 1 row and inserted 2, and the Type 1 `merge` (version 2) updated 1. The rerun added no version at all,
because neither `merge` found anything to change, so a retried load is harmless.

Because every Delta write is a new table version, the dimension as it was before the load is still readable.
**Time travel** (`version as of`, or `timestamp as of`) is how you answer "what did the dimension say when last
month's report ran?", and how you inspect or undo a bad load (`restore table dim_customer to version as of ...`).

In [ ]:
spark.sql("""
select 'before the load' as as_of, customer_sk, segment, is_current from dim_customer version as of 0 where customer_id = 101
union all
select 'now', customer_sk, segment, is_current from dim_customer where customer_id = 101
order by as_of, customer_sk
""").show(truncate=False)

**What to notice:** Version 0, the table as first built, has one current `smb` row for customer 101. The current version has the
closed `smb` row and the new `mid-market` row. Time travel works for as long as the table keeps its old files
(`vacuum` removes them after the retention period, 7 days by default), so it complements Type 2 history rather
than replacing it: Type 2 keeps history for years, and joins to it by date.

dbt's `snapshot` and Databricks' `AUTO CDC ... STORED AS SCD TYPE 2` (formerly `APPLY CHANGES`) generate this logic
for you; the rules (tracked columns, effective time, null-safe comparison, one atomic write) are the same.

## Types 3 and 6: previous and current values on each row

Type 2 answers "what was true then". Reports also ask "what is true now": revenue by the segment customers are
in *today*, including what they bought before they moved. With only Type 2 that needs a self-join to the
current row. **Type 6** stores it on every version as a `current_` column, overwritten (Type 1) whenever a new
version is added. **Type 3** stores a `previous_` column, which answers "what was it before the last change"
but nothing older.

**Where it matters**
- Dashboards that let users switch between "as it was" and "as it is now" views.

**Watch out for**
- Forgetting to update the `current_` column on old versions when a new version arrives.

### Compare past and current segments

Keep the segment valid at each fact’s date. Add columns for the prior and current segment where needed, then compare historical attribution with a report grouped by today’s segment.

In [ ]:
run("""
alter table dim_customer add columns (current_segment string, previous_segment string);

merge into dim_customer d                                        -- Type 6: today's value on every version
using (select customer_id, segment from dim_customer where is_current) c
on d.customer_id = c.customer_id
when matched then update set current_segment = c.segment;

merge into dim_customer d                                        -- Type 3: the value of the previous version
using (select customer_id, segment, valid_to from dim_customer) p
on d.customer_id = p.customer_id and p.valid_to = d.valid_from
when matched then update set previous_segment = p.segment;
""")
spark.sql("""
select customer_id, segment, current_segment, previous_segment, cast(valid_from as date) as valid_from, is_current
from dim_customer where customer_id in (101, 102) order by customer_sk
""").show(truncate=False)

In [ ]:
spark.sql("""
select d.segment as segment_at_time_of_order, d.current_segment, sum(f.net_amount) as net_sales
from fact_sales f
join dim_customer d
  on d.customer_id = f.customer_id and f.order_ts >= d.valid_from and f.order_ts < d.valid_to
group by all order by all
""").show(truncate=False)

**What to notice:** Every row for customer 101 now says `current_segment = mid-market`, including the old `smb` version. The report
shows both views from one join: customer 101's sales were made as `smb` but belong to `mid-market` today, and
customer 102's January 60.00 was `smb` at the time but is `enterprise` now. `previous_segment` is filled only on
rows that follow a change.

**Type 4** handles attributes that change often, such as a customer's credit band recalculated weekly. Versioning
the whole customer row on every recalculation would multiply the dimension. Instead the volatile attributes go into
a small **mini-dimension** of banded values (`dim_customer_profile`: credit band × age band × activity band) whose
key is stored on the fact, while the main dimension keeps the stable attributes.

## Testing a Type 2 dimension

A history table has invariants that a broken load violates quietly. Write each as a query that returns the
offending rows, so an empty result means the rule holds:

- exactly one current row per natural key;
- no overlapping versions for a key;
- no gaps between consecutive versions (when the history should be continuous);
- `valid_from < valid_to` on every row;
- surrogate keys unique.

Run them after every load and fail the pipeline when any returns rows (lesson 9 builds this into a test suite).

**Where it matters**
- After every dimension load, and in CI for the code that builds it.

**Watch out for**
- Tests that only count rows: they miss overlaps and gaps entirely.

### Find invalid customer histories

Run the checks on the clean dimension, then introduce a bad version. A failing query should return the offending key so you know which customer’s history to repair.

In [ ]:
TESTS = {
    "more than one current row": "select customer_id from dim_customer group by customer_id having count(*) filter (where is_current) <> 1",
    "overlapping versions": """select a.customer_id from dim_customer a join dim_customer b
                               on a.customer_id = b.customer_id and a.customer_sk < b.customer_sk
                              and a.valid_from < b.valid_to and b.valid_from < a.valid_to""",
    "gaps between versions": """select customer_id from (select customer_id, valid_from,
                                lag(valid_to) over (partition by customer_id order by valid_from) as prev_to from dim_customer)
                                where prev_to is not null and prev_to <> valid_from""",
    "empty or reversed range": "select customer_sk from dim_customer where valid_from >= valid_to",
}


def run_tests():
    for name, query in TESTS.items():
        bad = [row[0] for row in spark.sql(query).collect()]
        print(f"{'FAIL' if bad else 'ok  '}  {name}: {bad if bad else 'no rows'}")


run_tests()

In [ ]:
# simulate a load written as two statements (insert the new version, then close the old one) that failed in between
spark.sql("""
insert into dim_customer
select 99, customer_id, customer_name, email, city, 'enterprise', timestamp '2026-03-05', timestamp '9999-12-31', true,
       'enterprise', segment
from dim_customer where customer_id = 103
""")
run_tests()

**What to notice:** The real dimension passes all four tests. After the simulated half-finished load, customer 103 has two current
rows and two versions that overlap from 2026-03-05 onward, so a point-in-time join would double its later
sales. The gap test fails too: the old version still ends in 9999 instead of where the new one starts. The range
test still passes, which is why one test is not enough.

## Distinguish when a change applied from when it was known

**Effective time** says when a business value was valid. **Recorded time** says when the model knew about it. A late correction can change an effective interval in the past. Use both clocks when you need to reproduce a report exactly as it was known when published, a pattern called **bitemporal history**.

### Example

On March 10, the CRM reports that an enterprise upgrade took effect on February 1. For a February 15 sale, a February close prepared on March 1 saw SMB, while a report run after the correction sees enterprise. Store the recorded intervals as well as the effective intervals. Use half-open boundaries on each clock so adjacent versions cannot both match.

In [ ]:
history_comparison = spark.sql("""
with history(segment, effective_from, effective_to, recorded_from, recorded_to) as (
    select * from values
      ('smb', date '2026-01-01', date '9999-12-31', date '2026-01-01', date '2026-03-10'),
      ('smb', date '2026-01-01', date '2026-02-01', date '2026-03-10', date '9999-12-31'),
      ('enterprise', date '2026-02-01', date '9999-12-31', date '2026-03-10', date '9999-12-31')
), reports(published_on) as (
    select * from values (date '2026-03-01'), (date '2026-03-10')
)
select published_on, segment from reports join history
  on effective_from <= date '2026-02-15' and date '2026-02-15' < effective_to
 and recorded_from <= published_on and published_on < recorded_to
order by published_on
""")
assert [row.segment for row in history_comparison.collect()] == ["smb", "enterprise"]
history_comparison.show()

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A customer's phone number is corrected. Which SCD type should phone number use, and why?

**A.** Type 1. A correction should apply to all history; creating a new version would split the customer's facts across two rows for no reporting benefit.

**Q2.** Why is the current row's `valid_to` set to `9999-12-31` rather than null?

**A.** So every range condition works unchanged (`ts < valid_to`). A null would need `or valid_to is null` in every join, and forgetting it drops current rows.

**Q3.** A join of `fact_sales` to a Type 2 `dim_customer` returns more rows than the fact has. What are the two most likely causes?

**A.** The join is on the natural key without the date-range condition, or the dimension has overlapping versions for some keys (two current rows after a failed load).

**Q4.** Your load uses `d.city <> s.city` to detect changes. Which change will it miss?

**A.** A change to or from null: `null <> 'Leeds'` is null, not true. Use `is distinct from`.

**Q5.** Finance wants revenue by the region the customer is in today; sales operations wants it by the region at the time of sale. How do you serve both from one dimension?

**A.** Type 6: Type 2 rows with the region at the time, plus a `current_region` column on every row, overwritten when the region changes. Group by one or the other.

## Key takeaways

- Choose an SCD type per attribute: Type 1 for corrections, Type 2 when reports must keep the value that was true.
- Type 2 rows carry their own surrogate key and a half-open validity range ending in a far-future date.
- Join facts to the version whose range contains the fact's time; never with `between`, never on the natural key alone.
- Incremental loads compare with `is distinct from`, close and insert in one transaction, and must be idempotent.
- Type 6 adds a current-value column so one table answers "then" and "now"; Type 4 moves fast-changing attributes to a mini-dimension.
- Test one current row per key, no overlaps and no gaps after every load.

**Next:** Continue with [Data modeling 4: relationships and double counting](data-modeling-04-relationships-and-double-counting.ipynb).